# [4/4] SUMMARY – CNN vs RNN vs Transformer comparison report
### Run LAST, after the three branch notebooks have been committed. No GPU required (Accelerator: None).

**Loading the results:** right-hand panel → **+ Add Input → Your Work** → add the output of ALL THREE notebooks (01_CNN, 02_RNN, 03_Transformer). This notebook auto-discovers every `results/summary.csv` under `/kaggle/input/`.

In [ ]:
import glob, os, json
import pandas as pd
import matplotlib.pyplot as plt

# ===== 0. VERIFY that each input notebook version is a FULL RUN =====
# Quick-test/cancelled runs have every *_final stage at train_min ~1 min -> reject and ask for the right version.
state_files = sorted(glob.glob("/kaggle/input/**/state.json", recursive=True))
print("Found", len(state_files), "state.json files")
bad = []
for sf in state_files:
    slug = os.path.basename(os.path.dirname(sf))
    st = json.load(open(sf))
    fins = [v for k, v in st.items() if k.endswith("_final") and isinstance(v, dict)]
    maxmin = max((float(v.get("train_min", 0)) for v in fins), default=0)
    n_stage = len(st)
    # Detection notebooks (01/03) train for tens-to-hundreds of minutes -> threshold 3 min.
    # The RNN notebook (02) is light: full ~2 min/model, quick-test ~0.1-0.5 min -> threshold 1 min.
    has_det = any(k in st for k in ("yolo_final", "frcnn_final", "rtdetr_final"))
    thr = 3.0 if has_det else 1.0
    status = "OK (full run)" if maxmin >= thr else "❌ QUICK-TEST/INVALID"
    print(f"  {slug}: {n_stage} stages, max train_min={maxmin:.1f} (threshold {thr}) -> {status}")
    if maxmin < thr: bad.append(slug)
assert not bad, f"These inputs point at INVALID (quick-test) versions: {bad}. Open the Input panel -> select/pin the correct full-run version and re-run!"
assert len(state_files) >= 3, "You must Add Input the outputs of all three branch notebooks (01, 02, 03)!"

# ===== 1. Load the normalised results =====
paths = sorted(glob.glob("/kaggle/input/**/results/summary.csv", recursive=True))
print("summary.csv:", paths)
assert len(paths) >= 3, "Missing results/summary.csv — check the output version of each notebook!"
df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
display(df)

## 1. DETECTION comparison table (CNN vs Transformer) on TEST

In [ ]:
df_det = df[df.task=="detection"].set_index("model").sort_values("mAP50_test", ascending=False)
cols = [c for c in ["branch","cfg","mAP50_val","mAP50_test","mAP50_95_test","precision_test","recall_test","infer_ms","train_min"] if c in df_det.columns]
df_det = df_det[cols]
df_det.to_csv("/kaggle/working/final_detection_compare.csv")
display(df_det)

## 2. CLASSIFICATION comparison table, 12 classes (CNN vs RNN vs Transformer) on TEST

In [ ]:
df_cls = df[df.task=="classification"].set_index("model").sort_values("macro_f1", ascending=False)
cols = [c for c in ["branch","cfg","accuracy","macro_f1","worst_class_f1","train_min"] if c in df_cls.columns]
df_cls = df_cls[cols]
df_cls.to_csv("/kaggle/working/final_classification_compare.csv")
display(df_cls)

## 3. Charts + best-model selection

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,4.5))
df_det[["mAP50_test"]].astype(float).plot.bar(ax=axes[0], rot=15, legend=False, title="Detection – mAP@0.5 (TEST)")
df_cls[["macro_f1","worst_class_f1"]].astype(float).plot.bar(ax=axes[1], rot=15, title="Classification – macro-F1 & worst-class F1 (TEST)")
plt.tight_layout(); plt.savefig("/kaggle/working/final_compare.png", dpi=150); plt.show()

best_det = df_det["mAP50_test"].astype(float).idxmax()
best_cls = df_cls["macro_f1"].astype(float).idxmax()
best_branch_cls = df_cls.loc[best_cls, "branch"]
print(f">>> Best detector: {best_det}")
print(f">>> Best classifier: {best_cls} (branch {best_branch_cls})")
print(f">>> Recommended deployment pipeline: {best_det} + RNN temporal smoothing over the frame sequence.")

## 4. Comparison against the top 3 GitHub baselines (2023–2026)

In [ ]:
baseline_rows = [
    {"Source": f"This work – {best_det}", "Dataset": "PPE-Unified v1, 12 classes (10% test)",
     "mAP50": float(df_det.loc[best_det, "mAP50_test"]), "Notes": "Full HPO, fixed split seed 42, reproducible"},
    {"Source": "snehilsanyal/Construction-Site-Safety-PPE-Detection (YOLOv8n)", "Dataset": "Roboflow CSS, 2,801 images, 10 classes",
     "mAP50": None, "Notes": "No metrics published; only curves/confusion matrix"},
    {"Source": "jomarkow/Safety-Helmet-Detection (YOLOv8n)", "Dataset": "Kaggle Hard Hat 5k (different dataset)",
     "mAP50": None, "Notes": "No metrics published; 10-epoch proof of concept"},
    {"Source": "lyuwenyu/RT-DETR (R18 official)", "Dataset": "COCO val2017 (80 classes)",
     "mAP50": None, "Notes": "Official 46.5 mAP50-95 on COCO – reference only, different dataset"},
]
df_base = pd.DataFrame(baseline_rows)
df_base.to_csv("/kaggle/working/baseline_compare.csv", index=False)
display(df_base)

## 5. Generate the markdown report + README.md for GitHub

In [ ]:
hpo_files = sorted(glob.glob("/kaggle/input/**/results/hpo_*.csv", recursive=True))
hpo_md = ""
for p in hpo_files:
    name = os.path.basename(p).replace("hpo_","").replace(".csv","")
    hpo_md += f"\n### HPO – {name}\n\n" + pd.read_csv(p).to_markdown(index=False) + "\n"

report = f"""# Comparison report: CNN vs RNN vs Transformer – Safety-PPE detection, 12 classes

## 1. Dataset & protocol
- PPE-Unified v1 (three CC BY 4.0 sources merged, leakage removed by source image + MD5, normalised to 640×640), 12 classes = 6 compliant/violation pairs, 17,951 images.
- Fixed 70/20/10 split grouped by source image (group-aware, seed 42), shared by all three branches; TEST is used exactly once. Public: kaggle.com/datasets/quocdatlam/ppe-unified-v1
- Imbalance handling: repeat-factor oversampling (train), crop undersampling, class weights, model selection by macro-F1.

## 2. DETECTION results (TEST)
{df_det.to_markdown()}

## 3. CLASSIFICATION results, 12 classes (TEST)
{df_cls.to_markdown()}

## 4. Selected models
- Detection: **{best_det}** | Classification: **{best_cls}**
- Deployment pipeline: {best_det} + RNN temporal smoothing over the camera frame sequence.

## 5. Comparison against GitHub baselines, 2023–2026
{df_base.to_markdown(index=False)}

## 6. Appendix: hyperparameter screening tables
{hpo_md}
"""
with open("/kaggle/working/comparison_report.md","w",encoding="utf-8") as f: f.write(report)

readme = report.replace("# Comparison report", "# Helmet/PPE Detection – CNN vs RNN vs Transformer\n\n> Full comparison report")
with open("/kaggle/working/README.md","w",encoding="utf-8") as f: f.write(readme)
print("Exported: comparison_report.md, README.md, final_*.csv, final_compare.png")
print("Suggested GitHub repo layout: README.md + 4 notebooks + results/ (csv, png) + weights via GitHub Release")

## 6. Collect ALL outputs of the three branch notebooks into this notebook's output
Copies everything (state.json, results/, runs/ with weights, ...) EXCEPT the `dataset/` folder and the original pretrained `.pt` files — so downloading notebook 04's output alone gives every result without dragging along 2 GB of data.

In [ ]:
import shutil

EXCLUDE_DIRS = {"dataset", "__pycache__"}
collected = 0
for state in sorted(glob.glob("/kaggle/input/**/state.json", recursive=True)):
    src_root = os.path.dirname(state)
    slug = os.path.basename(src_root)
    dst_root = f"/kaggle/working/collected/{slug}"
    os.makedirs(dst_root, exist_ok=True)
    for item in os.listdir(src_root):
        if item in EXCLUDE_DIRS: continue
        if item.endswith(".pt"): continue   # original pretrained yolo26s.pt/rtdetr-l.pt, not a result
        s_, d_ = os.path.join(src_root, item), os.path.join(dst_root, item)
        if os.path.isdir(s_): shutil.copytree(s_, d_, dirs_exist_ok=True)
        else: shutil.copy2(s_, d_)
    collected += 1
    print("Collected:", slug, "->", dst_root)
print("Notebooks collected:", collected, "(need 3: notebooks 01, 02, 03)")
total = 0
for r, _, files in os.walk("/kaggle/working/collected"):
    total += sum(os.path.getsize(os.path.join(r, f)) for f in files)
print(f"Size of collected/: {total/1e9:.2f} GB")